# CLEAN Predictor Baselines On Metalloenzyme EC Splits

This notebook prepares and optionally trains the original CLEAN sequence-based EC predictor on the metalloenzyme train/test splits used for DeepMzyme.

It creates four CLEAN-predictor baselines:

1. `clean30_fold{fold}_metallo`: train CLEAN on extracted CLEAN30 metalloenzymes, test on extracted CLEAN30 metalloenzymes.
2. `clean30_fold{fold}_full`: train CLEAN on the full original CLEAN30 train split, test on the same extracted CLEAN30 metalloenzyme test set.
3. `care30_metallo`: train CLEAN on extracted CARE30 metalloenzymes, test on extracted CARE30 metalloenzymes.
4. `care30_full`: train CLEAN on the full original CARE Task 1 train set, test on the same extracted CARE30 metalloenzyme test set.

The notebook defaults to table preparation only. ESM-1b embedding generation, CLEAN training, inference, and scoring are long steps and are controlled by explicit flags in the config cell.

In [ ]:
from pathlib import Path
import csv
import json
import os
import re
import shutil
import subprocess
import sys
from typing import Iterable

import pandas as pd


def find_project_root(start: Path | None = None) -> Path:
    cur = (start or Path.cwd()).resolve()
    for path in [cur, *cur.parents]:
        if (path / "Plan.md").exists() and (path / "DeepMzyme_Data").exists():
            return path
    raise RuntimeError("Could not find DeepMzyme project root from current working directory")

PROJECT_ROOT = find_project_root()
WORK_ROOT = PROJECT_ROOT / "CLEAN" / "work"
PREPARED_ROOT = WORK_ROOT / "prepared_tables"
RESULTS_ROOT = WORK_ROOT / "scored_results"
OFFICIAL_CLEAN_DIR = WORK_ROOT / "official_CLEAN"
OFFICIAL_CLEAN_APP = OFFICIAL_CLEAN_DIR / "app"
CLEAN_REPO_URL = "https://github.com/tttianhao/CLEAN.git"

# Main experiment selector.
CLEAN_FOLD = 0

# Long-step flags. Keep these False until the normalized tables look correct.
RUN_INSTALL_CLEAN = False
INSTALL_REQUIREMENTS = False
RUN_GENERATE_ESM_AND_DISTANCES = False
RUN_TRAIN_CLEAN = False
RUN_INFERENCE = False
RUN_SCORE_RESULTS = False

# Do not use pre-MAHOMES CARE candidate sites as final metalloenzyme data unless explicitly debugging.
ALLOW_PROVISIONAL_CARE_INPUTS = False

# Official CLEAN recommends 2500 epochs for split30 triplet training. Reduce only for smoke tests.
TRIPLET_EPOCHS = 2500
TRIPLET_LR = 5e-4
PYTHON = sys.executable

for path in [WORK_ROOT, PREPARED_ROOT, RESULTS_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("WORK_ROOT:", WORK_ROOT)
print("OFFICIAL_CLEAN_APP:", OFFICIAL_CLEAN_APP)

## Data Normalization Helpers

Official CLEAN expects tab-delimited files named `data/<name>.csv` with columns:

```text
Entry    EC number    Sequence
```

The helpers below normalize the source CLEAN/CARE files and aggregate duplicate protein rows into one semicolon-separated EC label field per protein.

In [ ]:
EC_SPLIT_RE = re.compile(r"[;,]")
VALID_AA_RE = re.compile(r"^[A-Z*<>-]+$")


def read_table(path: Path) -> pd.DataFrame:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    # CLEAN source files are TSV with .csv extension; CARE and MAHOMES summaries are CSV.
    with path.open("r", encoding="utf-8", errors="ignore") as handle:
        first = handle.readline()
    sep = "\t" if "\t" in first and first.count("\t") >= first.count(",") else ","
    df = pd.read_csv(path, sep=sep, dtype=str, keep_default_na=False)
    unnamed = [c for c in df.columns if c.startswith("Unnamed") or c == ""]
    if unnamed:
        df = df.drop(columns=unnamed)
    return df


def first_existing_column(df: pd.DataFrame, candidates: Iterable[str]) -> str:
    lower = {c.lower(): c for c in df.columns}
    for candidate in candidates:
        if candidate in df.columns:
            return candidate
        if candidate.lower() in lower:
            return lower[candidate.lower()]
    raise KeyError(f"None of the candidate columns exist: {list(candidates)}; columns={list(df.columns)}")


def split_ecs(value: str) -> list[str]:
    ecs = []
    for item in EC_SPLIT_RE.split(str(value)):
        item = item.strip()
        if not item or item.lower() in {"nan", "none"}:
            continue
        ecs.append(item)
    return sorted(set(ecs))


def clean_sequence(seq: str) -> str:
    seq = str(seq).strip().replace(" ", "").replace("\n", "")
    return seq


def aggregate_rows(rows: Iterable[dict], *, context: str) -> pd.DataFrame:
    by_entry: dict[str, dict] = {}
    missing_sequence = []
    for row in rows:
        entry = str(row["Entry"]).strip()
        seq = clean_sequence(row.get("Sequence", ""))
        ecs = split_ecs(row.get("EC number", ""))
        if not entry or not ecs:
            continue
        if not seq:
            missing_sequence.append(entry)
            continue
        rec = by_entry.setdefault(entry, {"Entry": entry, "ECs": set(), "Sequence": seq})
        if rec["Sequence"] != seq:
            # Keep the first sequence; source split IDs should not have conflicting sequences.
            pass
        rec["ECs"].update(ecs)
    if missing_sequence:
        preview = ", ".join(sorted(set(missing_sequence))[:10])
        raise ValueError(f"{context}: missing sequence for {len(set(missing_sequence))} entries, e.g. {preview}")
    out = pd.DataFrame(
        {
            "Entry": rec["Entry"],
            "EC number": ";".join(sorted(rec["ECs"])),
            "Sequence": rec["Sequence"],
        }
        for rec in by_entry.values()
    )
    if out.empty:
        raise ValueError(f"{context}: no rows after normalization")
    return out.sort_values("Entry").reset_index(drop=True)


def normalize_sequence_source(path: Path, *, context: str) -> pd.DataFrame:
    df = read_table(path)
    entry_col = first_existing_column(df, ["Entry", "ID", "protein_id", "uniprot_id"])
    ec_col = first_existing_column(df, ["EC number", "EC", "EC All", "ecnumber"])
    seq_col = first_existing_column(df, ["Sequence", "Sequences", "sequence"])
    rows = (
        {"Entry": row[entry_col], "EC number": row[ec_col], "Sequence": row[seq_col]}
        for _, row in df.iterrows()
    )
    return aggregate_rows(rows, context=context)


def build_sequence_map(*source_paths: Path) -> dict[str, str]:
    seqs = {}
    for source in source_paths:
        if not Path(source).exists():
            continue
        df = normalize_sequence_source(source, context=f"sequence_map:{source.name}")
        seqs.update(dict(zip(df["Entry"], df["Sequence"])))
    return seqs


def normalize_metallo_summary(path: Path, *, sequence_sources: list[Path], context: str) -> pd.DataFrame:
    df = read_table(path)
    entry_col = first_existing_column(df, ["uniprot_id", "protein_id", "Entry", "ID"])
    ec_col = first_existing_column(df, ["ecnumber", "EC number", "EC", "EC All"])
    seq_map = build_sequence_map(*sequence_sources)
    rows = []
    missing = []
    for _, row in df.iterrows():
        entry = str(row[entry_col]).strip()
        seq = seq_map.get(entry, "")
        if not seq:
            missing.append(entry)
        rows.append({"Entry": entry, "EC number": row[ec_col], "Sequence": seq})
    if missing:
        preview = ", ".join(sorted(set(missing))[:10])
        raise ValueError(f"{context}: {len(set(missing))} entries not found in sequence sources, e.g. {preview}")
    return aggregate_rows(rows, context=context)


def write_clean_table(df: pd.DataFrame, name: str) -> Path:
    path = PREPARED_ROOT / f"{name}.csv"
    df[["Entry", "EC number", "Sequence"]].to_csv(path, sep="\t", index=False)
    return path


def summarize_clean_table(df: pd.DataFrame, name: str) -> dict:
    ec_lists = df["EC number"].map(split_ecs)
    ec1 = ec_lists.map(lambda xs: sorted({x.split(".")[0] for x in xs if x})).explode().value_counts().sort_index().to_dict()
    ec2 = ec_lists.map(lambda xs: sorted({".".join(x.split(".")[:2]) for x in xs if len(x.split(".")) >= 2})).explode().value_counts().sort_index().to_dict()
    return {
        "name": name,
        "proteins": int(len(df)),
        "unique_ec": int(len(set(e for xs in ec_lists for e in xs))),
        "ec1_counts": ec1,
        "ec2_classes": int(len(ec2)),
    }

## Prepare The Four CLEAN Predictor Jobs

CARE metalloenzyme train/test tables require the CARE AlphaFill + MAHOMES2 export. If that export is not finished yet, the notebook prepares only the CLEAN30 jobs unless `ALLOW_PROVISIONAL_CARE_INPUTS` is set to `True`.

In [ ]:
# Source paths.
clean_full_train = PROJECT_ROOT / f"DeepMzyme_Data/CLEAN_all_train_valid_splits/split30/split30_train_split_{CLEAN_FOLD}.csv"
clean_full_test = PROJECT_ROOT / f"DeepMzyme_Data/CLEAN_all_train_valid_splits/split30/split30_test_split_{CLEAN_FOLD}_curate.csv"
clean_metallo_train = PROJECT_ROOT / f"DeepMzyme_Data/CLEAN_30_train_test_split_{CLEAN_FOLD}/train/final_data_summarazing_table_transition_metals_only_catalytic.csv"
clean_metallo_test = PROJECT_ROOT / f"DeepMzyme_Data/CLEAN_30_train_test_split_{CLEAN_FOLD}/test/final_data_summarazing_table_transition_metals_only_catalytic.csv"

care_full_train = PROJECT_ROOT / "DeepMzyme_Data/CARE_dataset/CARE_datasets/splits/task1/protein_train.csv"
care_full_test = PROJECT_ROOT / "DeepMzyme_Data/CARE_dataset/CARE_datasets/splits/task1/30_protein_test.csv"
care_export_root = Path("/media/Data/care_sets/task1_30_clusterRes30/exported/CARE_task1_30_clusterRes30_train_test_metallo")
care_metallo_train_final = care_export_root / "train/final_data_summarazing_table_transition_metals_only_catalytic.csv"
care_metallo_test_final = care_export_root / "test/final_data_summarazing_table_transition_metals_only_catalytic.csv"
care_metallo_train_provisional = Path("/media/Data/care_sets/task1_30_clusterRes30/mahomes_inputs/train/candidate_site_summary.csv")
care_metallo_test_provisional = Path("/media/Data/care_sets/task1_30_clusterRes30/mahomes_inputs/test/candidate_site_summary.csv")

prepared = {}
summaries = []

# CLEAN30 full and metalloenzyme tables.
clean30_full_train_name = f"clean30_fold{CLEAN_FOLD}_full_train"
clean30_metallo_train_name = f"clean30_fold{CLEAN_FOLD}_metallo_train"
clean30_metallo_test_name = f"clean30_fold{CLEAN_FOLD}_metallo_test"

clean30_full_train_df = normalize_sequence_source(clean_full_train, context=clean30_full_train_name)
clean30_metallo_train_df = normalize_metallo_summary(
    clean_metallo_train,
    sequence_sources=[clean_full_train, clean_full_test],
    context=clean30_metallo_train_name,
)
clean30_metallo_test_df = normalize_metallo_summary(
    clean_metallo_test,
    sequence_sources=[clean_full_test, clean_full_train],
    context=clean30_metallo_test_name,
)

for name, df in [
    (clean30_full_train_name, clean30_full_train_df),
    (clean30_metallo_train_name, clean30_metallo_train_df),
    (clean30_metallo_test_name, clean30_metallo_test_df),
]:
    prepared[name] = write_clean_table(df, name)
    summaries.append(summarize_clean_table(df, name))

# CARE full train is independent of MAHOMES. CARE metallo tables wait for final export by default.
care30_full_train_name = "care30_full_train"
care30_metallo_train_name = "care30_clusterRes30_metallo_train"
care30_metallo_test_name = "care30_metallo_test"

care30_full_train_df = normalize_sequence_source(care_full_train, context=care30_full_train_name)
prepared[care30_full_train_name] = write_clean_table(care30_full_train_df, care30_full_train_name)
summaries.append(summarize_clean_table(care30_full_train_df, care30_full_train_name))

care_metallo_ready = care_metallo_train_final.exists() and care_metallo_test_final.exists()
care_metallo_source_status = "final_mahomes_export" if care_metallo_ready else "missing_final_mahomes_export"
if not care_metallo_ready and ALLOW_PROVISIONAL_CARE_INPUTS:
    care_metallo_ready = care_metallo_train_provisional.exists() and care_metallo_test_provisional.exists()
    care_metallo_source_status = "PROVISIONAL_pre_mahomes_candidate_sites" if care_metallo_ready else care_metallo_source_status
    care_metallo_train_path = care_metallo_train_provisional
    care_metallo_test_path = care_metallo_test_provisional
else:
    care_metallo_train_path = care_metallo_train_final
    care_metallo_test_path = care_metallo_test_final

if care_metallo_ready:
    care30_metallo_train_df = normalize_metallo_summary(
        care_metallo_train_path,
        sequence_sources=[care_full_train, care_full_test],
        context=care30_metallo_train_name,
    )
    care30_metallo_test_df = normalize_metallo_summary(
        care_metallo_test_path,
        sequence_sources=[care_full_test, care_full_train],
        context=care30_metallo_test_name,
    )
    for name, df in [
        (care30_metallo_train_name, care30_metallo_train_df),
        (care30_metallo_test_name, care30_metallo_test_df),
    ]:
        prepared[name] = write_clean_table(df, name)
        summaries.append(summarize_clean_table(df, name))
else:
    print("CARE metalloenzyme final export is not ready yet; CARE metallo jobs will be skipped for now.")
    print("Expected final train:", care_metallo_train_final)
    print("Expected final test:", care_metallo_test_final)

# Build job manifest. Each run gets its own copy of the test CSV name because official CLEAN writes results by test_data name.
jobs = [
    {
        "job_name": f"clean30_fold{CLEAN_FOLD}_metallo",
        "train_data": clean30_metallo_train_name,
        "test_source_data": clean30_metallo_test_name,
        "test_data": f"clean30_fold{CLEAN_FOLD}_metallo_test__for_clean30_metallo_train",
        "model_name": f"clean30_fold{CLEAN_FOLD}_metallo_triplet",
        "benchmark": "CLEAN30",
        "train_scope": "metalloenzyme_only",
        "test_scope": "metalloenzyme_only",
    },
    {
        "job_name": f"clean30_fold{CLEAN_FOLD}_full",
        "train_data": clean30_full_train_name,
        "test_source_data": clean30_metallo_test_name,
        "test_data": f"clean30_fold{CLEAN_FOLD}_metallo_test__for_clean30_full_train",
        "model_name": f"clean30_fold{CLEAN_FOLD}_full_triplet",
        "benchmark": "CLEAN30",
        "train_scope": "full_original_train_split",
        "test_scope": "metalloenzyme_only",
    },
]

if care_metallo_ready:
    jobs.extend([
        {
            "job_name": "care30_metallo",
            "train_data": care30_metallo_train_name,
            "test_source_data": care30_metallo_test_name,
            "test_data": "care30_metallo_test__for_care30_metallo_train",
            "model_name": "care30_metallo_triplet",
            "benchmark": "CARE30",
            "train_scope": "metalloenzyme_only_clusterRes30",
            "test_scope": "metalloenzyme_only",
        },
        {
            "job_name": "care30_full",
            "train_data": care30_full_train_name,
            "test_source_data": care30_metallo_test_name,
            "test_data": "care30_metallo_test__for_care30_full_train",
            "model_name": "care30_full_triplet",
            "benchmark": "CARE30",
            "train_scope": "full_original_task1_train",
            "test_scope": "metalloenzyme_only",
        },
    ])

# Duplicate test tables under run-specific names to avoid official CLEAN result overwrite.
for job in jobs:
    source_path = prepared[job["test_source_data"]]
    target_path = PREPARED_ROOT / f"{job['test_data']}.csv"
    shutil.copy2(source_path, target_path)
    prepared[job["test_data"]] = target_path

manifest = pd.DataFrame(jobs)
manifest["care_metallo_source_status"] = care_metallo_source_status
manifest_path = PREPARED_ROOT / "jobs_manifest.csv"
summary_path = PREPARED_ROOT / "table_summary.json"
manifest.to_csv(manifest_path, index=False)
summary_path.write_text(json.dumps(summaries, indent=2, sort_keys=True), encoding="utf-8")

print("Prepared tables:")
for name, path in sorted(prepared.items()):
    print(f"  {name}: {path}")
print("\nJob manifest:", manifest_path)
display(manifest)
print("\nTable summaries:")
print(summary_path.read_text())

## Install Official CLEAN Code

This clones the official CLEAN implementation into `CLEAN/work/official_CLEAN` and installs it from its `app/` directory.

Run this only in the environment where you want to train CLEAN. Official CLEAN was developed with Python 3.10 and ESM-1b; dependency conflicts with the DeepMzyme environment are possible, so a separate environment is preferable for serious CLEAN runs.

In [ ]:
if RUN_INSTALL_CLEAN:
    if not OFFICIAL_CLEAN_DIR.exists():
        subprocess.run(["git", "clone", CLEAN_REPO_URL, str(OFFICIAL_CLEAN_DIR)], check=True)
    if INSTALL_REQUIREMENTS:
        subprocess.run([PYTHON, "-m", "pip", "install", "-r", "requirements.txt"], cwd=OFFICIAL_CLEAN_APP, check=True)
    subprocess.run([PYTHON, "build.py", "install"], cwd=OFFICIAL_CLEAN_APP, check=True)
    esm_dir = OFFICIAL_CLEAN_APP / "esm"
    if not esm_dir.exists():
        subprocess.run(["git", "clone", "https://github.com/facebookresearch/esm.git", str(esm_dir)], check=True)
    (OFFICIAL_CLEAN_APP / "data" / "esm_data").mkdir(parents=True, exist_ok=True)
    (OFFICIAL_CLEAN_APP / "data" / "distance_map").mkdir(parents=True, exist_ok=True)
    (OFFICIAL_CLEAN_APP / "data" / "model").mkdir(parents=True, exist_ok=True)
else:
    print("RUN_INSTALL_CLEAN is False; skipped official CLEAN clone/install.")

## Sync Prepared Tables Into Official CLEAN

This copies the normalized train/test tables into `official_CLEAN/app/data/`.

In [ ]:
def load_jobs() -> list[dict]:
    manifest_path = PREPARED_ROOT / "jobs_manifest.csv"
    if not manifest_path.exists():
        raise FileNotFoundError("Run the preparation cell first: " + str(manifest_path))
    return pd.read_csv(manifest_path).to_dict("records")


def sync_tables_to_clean_app() -> None:
    if not OFFICIAL_CLEAN_APP.exists():
        raise FileNotFoundError(f"Official CLEAN app not found: {OFFICIAL_CLEAN_APP}. Set RUN_INSTALL_CLEAN=True first or clone it manually.")
    data_dir = OFFICIAL_CLEAN_APP / "data"
    data_dir.mkdir(parents=True, exist_ok=True)
    jobs = load_jobs()
    names = set()
    for job in jobs:
        names.add(job["train_data"])
        names.add(job["test_data"])
    for name in sorted(names):
        src = PREPARED_ROOT / f"{name}.csv"
        dst = data_dir / f"{name}.csv"
        if not src.exists():
            raise FileNotFoundError(src)
        shutil.copy2(src, dst)
        print("copied", src, "->", dst)

if OFFICIAL_CLEAN_APP.exists():
    sync_tables_to_clean_app()
else:
    print("Official CLEAN app is not present yet; run install cell before syncing tables.")

## Generate ESM-1b Embeddings And Distance Maps

This is the first expensive step. It downloads/uses the ESM-1b model and writes per-protein embeddings to `official_CLEAN/app/data/esm_data/`.

For every training table, official CLEAN also mutates orphan EC sequences and computes distance maps.

In [ ]:
if RUN_GENERATE_ESM_AND_DISTANCES:
    if not OFFICIAL_CLEAN_APP.exists():
        raise FileNotFoundError(OFFICIAL_CLEAN_APP)
    sys.path.insert(0, str(OFFICIAL_CLEAN_APP / "src"))
    old_cwd = Path.cwd()
    os.chdir(OFFICIAL_CLEAN_APP)
    try:
        from CLEAN.utils import csv_to_fasta, retrive_esm1b_embedding, mutate_single_seq_ECs, compute_esm_distance, ensure_dirs

        ensure_dirs("data/esm_data")
        ensure_dirs("data/distance_map")
        jobs = load_jobs()
        train_names = sorted({job["train_data"] for job in jobs})
        test_names = sorted({job["test_data"] for job in jobs})
        all_names = sorted(set(train_names) | set(test_names))

        for name in all_names:
            print("[FASTA]", name)
            csv_to_fasta(f"data/{name}.csv", f"data/{name}.fasta")
            print("[ESM]", name)
            retrive_esm1b_embedding(name)

        for name in train_names:
            print("[MUTATE SINGLE-EC]", name)
            mutated_fasta_name = mutate_single_seq_ECs(name)
            print("[ESM MUTATED]", mutated_fasta_name)
            retrive_esm1b_embedding(mutated_fasta_name)
            print("[DISTANCE]", name)
            compute_esm_distance(name)
    finally:
        os.chdir(old_cwd)
else:
    print("RUN_GENERATE_ESM_AND_DISTANCES is False; skipped ESM/distance generation.")

## Train CLEAN Triplet Models

This runs the official `train-triplet.py` for each job's training set. The full CARE training baseline is expected to be much slower and larger than the metalloenzyme-only jobs.

In [ ]:
if RUN_TRAIN_CLEAN:
    jobs = load_jobs()
    for job in jobs:
        cmd = [
            PYTHON,
            "train-triplet.py",
            "--training_data", job["train_data"],
            "--model_name", job["model_name"],
            "--epoch", str(TRIPLET_EPOCHS),
            "--learning_rate", str(TRIPLET_LR),
        ]
        print("[TRAIN]", job["job_name"], " ".join(cmd))
        subprocess.run(cmd, cwd=OFFICIAL_CLEAN_APP, check=True)
else:
    print("RUN_TRAIN_CLEAN is False; skipped CLEAN training.")

## Run CLEAN Max-Separation Inference

Official CLEAN writes results as `official_CLEAN/app/results/<test_data>_maxsep.csv`. The notebook uses a run-specific duplicate test name for each job so results are not overwritten.

In [ ]:
if RUN_INFERENCE:
    jobs = load_jobs()
    for job in jobs:
        infer_code = (
            "from CLEAN.infer import infer_maxsep\n"
            f"infer_maxsep({job['train_data']!r}, {job['test_data']!r}, "
            f"report_metrics=True, pretrained=False, model_name={job['model_name']!r})\n"
        )
        print("[INFER]", job["job_name"])
        subprocess.run([PYTHON, "-c", infer_code], cwd=OFFICIAL_CLEAN_APP, check=True)
else:
    print("RUN_INFERENCE is False; skipped CLEAN inference.")

## Score EC1/EC2 Results

This scorer treats EC labels as multi-label sets. For each protein and EC level, the main top-1 metric is correct if CLEAN's first predicted EC prefix is in the true EC-prefix set.

It also reports multi-label macro F1 and macro recall using the predicted top-1 prefix set.

In [ ]:
def ec_prefix(ec: str, level: int) -> str | None:
    ec = str(ec).strip().replace("EC:", "")
    if not ec:
        return None
    parts = ec.split(".")
    if len(parts) < level:
        return None
    return ".".join(parts[:level])


def parse_clean_predictions(path: Path) -> dict[str, list[str]]:
    preds = {}
    with Path(path).open("r", encoding="utf-8", errors="ignore") as handle:
        reader = csv.reader(handle)
        for row in reader:
            if not row:
                continue
            entry = row[0].strip()
            ec_preds = []
            for item in row[1:]:
                item = item.strip()
                if not item.startswith("EC:"):
                    continue
                ec = item.split("/", 1)[0].replace("EC:", "").strip()
                if ec:
                    ec_preds.append(ec)
            preds[entry] = ec_preds
    return preds


def score_predictions(truth_csv: Path, prediction_csv: Path, *, levels=(1, 2)) -> pd.DataFrame:
    from sklearn.metrics import f1_score, recall_score
    from sklearn.preprocessing import MultiLabelBinarizer

    truth = pd.read_csv(truth_csv, sep="\t", dtype=str, keep_default_na=False)
    preds = parse_clean_predictions(prediction_csv)
    rows = []
    for level in levels:
        true_sets = []
        pred_top1_sets = []
        any_true_hits = []
        missing_predictions = 0
        for _, row in truth.iterrows():
            entry = row["Entry"]
            true_prefixes = {ec_prefix(ec, level) for ec in split_ecs(row["EC number"])}
            true_prefixes = {x for x in true_prefixes if x}
            pred_ecs = preds.get(entry, [])
            if not pred_ecs:
                missing_predictions += 1
                pred_prefixes = set()
            else:
                top1 = ec_prefix(pred_ecs[0], level)
                pred_prefixes = {top1} if top1 else set()
            true_sets.append(true_prefixes)
            pred_top1_sets.append(pred_prefixes)
            any_true_hits.append(bool(true_prefixes & pred_prefixes))
        labels = sorted(set().union(*true_sets, *pred_top1_sets))
        if labels:
            mlb = MultiLabelBinarizer(classes=labels)
            y_true = mlb.fit_transform(true_sets)
            y_pred = mlb.transform(pred_top1_sets)
            macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
            micro_f1 = f1_score(y_true, y_pred, average="micro", zero_division=0)
            macro_recall = recall_score(y_true, y_pred, average="macro", zero_division=0)
        else:
            macro_f1 = micro_f1 = macro_recall = 0.0
        rows.append({
            "ec_level": level,
            "n_test_proteins": int(len(truth)),
            "n_true_classes": int(len(set().union(*true_sets))) if true_sets else 0,
            "missing_predictions": int(missing_predictions),
            "top1_any_true_accuracy": float(sum(any_true_hits) / len(any_true_hits)) if any_true_hits else 0.0,
            "macro_f1_top1": float(macro_f1),
            "micro_f1_top1": float(micro_f1),
            "macro_recall_top1": float(macro_recall),
        })
    return pd.DataFrame(rows)


def score_all_jobs() -> pd.DataFrame:
    jobs = load_jobs()
    all_metrics = []
    for job in jobs:
        truth_csv = OFFICIAL_CLEAN_APP / "data" / f"{job['test_data']}.csv"
        pred_csv = OFFICIAL_CLEAN_APP / "results" / f"{job['test_data']}_maxsep.csv"
        if not pred_csv.exists():
            print("[SKIP missing prediction]", pred_csv)
            continue
        metrics = score_predictions(truth_csv, pred_csv)
        for key, value in job.items():
            metrics[key] = value
        out_dir = RESULTS_ROOT / job["job_name"]
        out_dir.mkdir(parents=True, exist_ok=True)
        metrics.to_csv(out_dir / "ec_level_metrics.csv", index=False)
        shutil.copy2(pred_csv, out_dir / pred_csv.name)
        all_metrics.append(metrics)
    if not all_metrics:
        return pd.DataFrame()
    combined = pd.concat(all_metrics, ignore_index=True)
    combined.to_csv(RESULTS_ROOT / "all_clean_predictor_metrics.csv", index=False)
    return combined

if RUN_SCORE_RESULTS:
    combined_metrics = score_all_jobs()
    display(combined_metrics)
else:
    print("RUN_SCORE_RESULTS is False; skipped scoring.")

## Expected Final Outputs

After all long flags are enabled and completed, the important files are:

```text
CLEAN/work/prepared_tables/jobs_manifest.csv
CLEAN/work/prepared_tables/table_summary.json
CLEAN/work/official_CLEAN/app/data/model/<model_name>.pth
CLEAN/work/official_CLEAN/app/results/<test_data>_maxsep.csv
CLEAN/work/scored_results/all_clean_predictor_metrics.csv
CLEAN/work/scored_results/<job_name>/ec_level_metrics.csv
```

Use `all_clean_predictor_metrics.csv` to compare CLEAN-matched and CLEAN-full against DeepMzyme on the same metalloenzyme-only test proteins.